# Fine-tune Gremlin's primary (free, on Colab's T4)

**Why Colab, not the desktop:** loading a 7-8B model's full-precision weights before quantizing needs ~14-16GB of System RAM. The desktop has 7.5GB -- that's not a VRAM problem (the 2070 Super's 8GB is fine for QLoRA itself), it's a RAM problem: the OS swaps the overflow to the HDD, swap-thrashes, and the box locks up (this happened for real, twice, on 2026-09-08). Colab's free tier gives 16GB System RAM + a free T4 GPU -- the exact thing that's missing locally, and it costs nothing.

**What this trains:** a LoRA adapter for `mlabonne/Meta-Llama-3.1-8B-Instruct-abliterated` -- the exact model Gremlin's `llama-3.1-8b-abliterated` GGUF was quantized from, not a different base. That match matters: a LoRA trained on a different model (even a very similar one) isn't guaranteed to blend cleanly at inference time.

**Hyperparameters mirror `gremlin_core/finetune.py`'s already-tuned gentle defaults exactly** (epochs=1, lr=1e-4, LoRA r=8/alpha=16/dropout=0.1, cosine warmup, grad-clip 0.3) -- these were dialed in after an earlier 3-epoch/lr-2e-4 run on the 3B model came out WORSE (repetition loops, garbled facts). One clean pass that nudges the base, not one that overwrites it.

**Before running this notebook**, build the training data on the desktop (safe, local, no RAM risk -- this step doesn't touch the model at all):
```bash
cd ~/Downloads/gremlin
venv/bin/python -c "from gremlin_core import finetune; print(finetune.write_training_set('.'))"
```
That writes `data/training_set.jsonl` and `data/eval_set.jsonl`. More rows = a better run; do this right before you open this notebook, not weeks early.

**Runtime > Change runtime type > T4 GPU**, then run the cells top to bottom.

In [ ]:
%%capture
# Unsloth's own recommended Colab install. If this errors, check
# https://github.com/unslothai/unsloth for the current one-liner --
# this is a fast-moving library and install steps do drift over time.
!pip install unsloth
!pip install --no-deps trl peft accelerate bitsandbytes

In [ ]:
from unsloth import FastLanguageModel
import torch

# 2048 not the desktop's forced-down 384 -- the T4's 16GB VRAM has real
# headroom that the 2070 Super's 8GB (already holding the 4-bit base)
# doesn't, so answers aren't truncated nearly as hard during training.
max_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="mlabonne/Meta-Llama-3.1-8B-Instruct-abliterated",
    max_seq_length=max_seq_length,
    load_in_4bit=True,
    dtype=None,  # auto-detect (bfloat16 on a T4)
)

In [ ]:
# Same shape as gremlin_core/finetune.py's LoraConfig -- r=8/alpha=16,
# attention projections only, gentle by design (a small adapter can
# only nudge the base, not overwrite its general ability).
model = FastLanguageModel.get_peft_model(
    model,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

## Upload the training data
Pick `training_set.jsonl` and `eval_set.jsonl` from `~/Downloads/gremlin/data/` on the desktop (built by the step in the intro cell above).

In [ ]:
from google.colab import files
uploaded = files.upload()
assert "training_set.jsonl" in uploaded, "upload training_set.jsonl (and eval_set.jsonl if you have one)"
has_eval = "eval_set.jsonl" in uploaded

In [ ]:
import json
from datasets import Dataset

def _load_jsonl(path):
    rows = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

def _to_text(example):
    # Same call the desktop pipeline uses -- renders the {"messages": [...]}
    # rows through the model's own chat template, not a hand-rolled format.
    return {"text": tokenizer.apply_chat_template(
        example["messages"], tokenize=False, add_generation_prompt=False)}

train_rows = _load_jsonl("training_set.jsonl")
print(f"{len(train_rows)} training examples")
train_ds = Dataset.from_list(train_rows).map(_to_text, remove_columns=["messages"])

eval_ds = None
if has_eval:
    eval_rows = _load_jsonl("eval_set.jsonl")
    print(f"{len(eval_rows)} eval examples")
    if eval_rows:
        eval_ds = Dataset.from_list(eval_rows).map(_to_text, remove_columns=["messages"])

In [ ]:
from trl import SFTTrainer, SFTConfig
from transformers import EarlyStoppingCallback

# Every value here matches gremlin_core/finetune.py's train_lora()
# defaults exactly -- see that file's comments for why each one is
# what it is. gradient_accumulation_steps stays at 16 (not the 4 a
# generic guide might suggest) because that's this project's own
# already-tested value, not a guess.
args = SFTConfig(
    output_dir="outputs",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=1e-4,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    weight_decay=0.01,
    max_grad_norm=0.3,
    bf16=True,
    logging_steps=5,
    save_strategy="epoch" if eval_ds is not None else "no",
    save_total_limit=2,
    load_best_model_at_end=eval_ds is not None,
    metric_for_best_model="eval_loss",
    eval_strategy="epoch" if eval_ds is not None else "no",
    optim="paged_adamw_8bit",
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    report_to=[],
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    args=args,
    # stop the moment held-out loss stops improving -- don't keep
    # grinding the same handful of examples into the weights
    callbacks=([EarlyStoppingCallback(early_stopping_patience=2)] if eval_ds is not None else []),
)

result = trainer.train()
print(result)

## Export
This does the merge (base + adapter) AND the GGUF conversion+quantization in one call -- something the desktop specifically avoids doing locally (the merge step alone needs ~15-20GB RAM). Colab has the room; the desktop doesn't. Result is ONE self-contained GGUF, no `lora_path` complexity needed.

In [ ]:
model.save_pretrained_gguf("gremlin-llama31-8b-abliterated-ft", tokenizer, quantization_method="q4_k_m")

In [ ]:
import glob
from google.colab import files

gguf_files = glob.glob("gremlin-llama31-8b-abliterated-ft/*.gguf")
print("found:", gguf_files)
for f in gguf_files:
    files.download(f)  # saves to your phone/computer's Downloads

## Bring it home

1. Move the downloaded `.gguf` into `~/Downloads/gremlin/models/` on the desktop.
2. Add this block to `config/models.yaml` under `models:` (matches the pattern every other entry in that file uses -- q8_0 KV, not q4_0: this repo confirmed live on 2026-09-13 that q4_0 degenerates Llama/Qwen-family GGUFs into repetition garbage):

```yaml
  - name: gremlin-llama31-8b-abliterated-ft
    type: local_gguf
    display_name: "Llama-3.1-8B Abliterated (fine-tuned)"
    model_path: "/home/mickey/Downloads/gremlin/models/<the file you downloaded>.gguf"
    n_ctx: 16384
    n_gpu_layers: -1
    flash_attn: true
    kv_cache_type: q8_0
    chat_format: llama-3
```

3. **A/B it before trusting it -- same discipline as every other model swap this project has done:**
```
/model switch gremlin-llama31-8b-abliterated-ft
```
then `systemctl --user restart gremlin.service`, and ask it the same handful of things you'd ask the current primary. If it's not clearly better (more like your voice, no worse at anything), `/model switch llama-3.1-8b-abliterated` and restart puts it back exactly as it was -- nothing about this process touches the desktop's config until you deliberately bring a file back and add it.